In [13]:
import dill
import h5py
import json
import torch
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from pina import LabelTensor

Load del modello addestrato

In [14]:
# Lavoriamo con i double
torch.set_default_dtype(torch.float64)

In [15]:
# Funzione per caricare il modello addestrato
def load_state(file_name):
    """
    Load object saved attributes from file_name
    """
    return dill.load(open(file_name, "rb"))

In [16]:
# Esempio load
loaded_trainer = load_state("Files/3_Base_pinn_temp.pkl")

out_prova = loaded_trainer.solver.neural_net(LabelTensor(torch.tensor([[0, 0.1, 0.2, 0.3]]), labels=["x", "y", "z", "t"]))
print(out_prova)

labels(['u'])
LabelTensor([[[20.2921]]], grad_fn=<AliasBackward0>)


### Calcolo errore rispetto a tutti i punti

In [17]:
# Load di Solution.csv
sol_file_path = "./Files/2_Base_FEM_Solution.csv"
df = pd.read_csv(sol_file_path, sep=";", index_col=False)

print(df.head())

         x1        x2        x3    t          u
0  0.896930  1.078061 -1.285433  0.0  20.026726
1  0.736314 -1.066762 -1.205756  0.0  20.026726
2 -1.052088 -0.954513 -1.183357  0.0  20.026726
3 -0.692441  0.835835 -1.280725  0.0  20.026726
4  1.303753  1.023542  1.066453  0.0  20.026726


In [18]:
# Creazione Label Tensor
pts_test = LabelTensor(
    torch.tensor(df.iloc[:, :4].values, dtype=torch.float64),
    labels=["x", "y", "z", "t"]
)

In [19]:
# Calcolo approssimazioni
u_tilde = loaded_trainer.solver.neural_net(pts_test).tensor.detach().numpy()

Calcolo errori

In [20]:
u_real = df.iloc[:, -1].values
u_real = u_real.reshape((u_real.shape[0], 1))

err_assoluto = np.linalg.norm(u_real-u_tilde, ord=2)
err_relative = err_assoluto/np.linalg.norm(u_real, ord=2)

print("TOTAL")
print(f"Assoluto = {err_assoluto:.2e}\tRelativo = {err_relative:.2e}")

TOTAL
Assoluto = 1.13e+03	Relativo = 4.22e-02


In [21]:
name_file = "Files/3_Base_Errore_tot.json"

tot_err = {
    "assolute" : err_assoluto,
    "relative" : err_relative
}

with open(name_file, "w") as outfile:
    json.dump(tot_err, outfile)

# Scrittura file .xdmf

## Modifica file .h5

In [22]:
# Opening JSON file
f = open('Files/2_Base_info_solution_fem.json')

# returns JSON object as a dictionary
data_fem = json.load(f)

# Chiusura del file
f.close()

tot_special_key = data_fem["steps"]
grid_dimension = data_fem["single_grid_dimension"]

In [23]:
# Percorso del file HDF5 di origine
source_h5_file_path = "Files/2_Base_FEM_Solution.h5"
# Percorso del nuovo file HDF5
destination_h5_file_path = "Files/3_Base_PINN_solution.h5"

# Chiavi speciali per cui inserire nuovi dati
special_keys = [str(i) for i in range(tot_special_key)]

def copy_h5_structure_and_data(source_group, destination_group, special_keys=None):
    """
    Copia ricorsivamente la struttura ad albero e i dati da un gruppo sorgente a un gruppo di destinazione in un file HDF5.
    Per le chiavi speciali specificate, crea un array di numeri casuali con le stesse dimensioni del dataset originale.

    Args:
        source_group (h5py.Group): Gruppo di origine da cui copiare la struttura ad albero e i dati.
        destination_group (h5py.Group): Gruppo di destinazione in cui copiare la struttura ad albero e i dati.
        special_keys (list): Lista di chiavi speciali per le quali creare array di numeri casuali. Default è None.

    """
    if special_keys is None:
        special_keys = []
        
    for key in source_group.keys():
        if isinstance(source_group[key], h5py.Group):
            # Se l'elemento è un gruppo, crea un gruppo corrispondente nella destinazione e copia ricorsivamente la struttura ad albero e i dati
            new_group = destination_group.create_group(key)
            copy_h5_structure_and_data(source_group[key], new_group, special_keys)
        elif isinstance(source_group[key], h5py.Dataset):
            # Se l'elemento è un dataset, copia i dati nel nuovo file
            source_dataset = source_group[key]
            if key in special_keys:
                # Se la chiave è nelle chiavi speciali, crea un array di numeri casuali con le stesse dimensioni del dataset originale
                random_data = u_tilde[(int(key)*grid_dimension):((int(key)+1)*grid_dimension)].reshape(grid_dimension,)
                destination_group.create_dataset(key, data=random_data)
            else:
                # Altrimenti, copia i dati dal dataset originale
                data = source_dataset[()]
                destination_group.create_dataset(key, data=data)


# Apri il file HDF5 di origine in modalità di sola lettura
with h5py.File(source_h5_file_path, "r") as source_hdf5_file:
    # Apri il nuovo file HDF5 in modalità scrittura
    with h5py.File(destination_h5_file_path, "w") as destination_hdf5_file:
        # Copia la struttura ad albero e i dati dal file di origine al nuovo file
        copy_h5_structure_and_data(source_hdf5_file, destination_hdf5_file, special_keys)

        print("Struttura ad albero e dati copiati con successo da {} a {}.".format(source_h5_file_path, destination_h5_file_path))

Struttura ad albero e dati copiati con successo da Files/2_Base_FEM_Solution.h5 a Files/3_Base_PINN_solution.h5.


Creazione file -.xdmf

In [24]:
# Percorso del file XDMF esistente
file_path = "Files/2_Base_FEM_Solution.xdmf"

# Analizza il file XDMF esistente utilizzando ElementTree
tree = ET.parse(file_path)
root = tree.getroot()

# Funzione per sostituire i riferimenti ai file .h5 con un nuovo file
def replace_h5_references(element, new_h5_file):
    if element.tag == "DataItem" and "Format" in element.attrib and element.attrib["Format"] == "HDF":
        # Ottieni il percorso del file .h5 originale
        old_h5_path = element.text
        # Estrai il percorso relativo alla struttura nel nuovo file .h5
        relative_path = old_h5_path.split(":")[1]
        # Sostituisci il nome del file .h5 con il nuovo file mantenendo la struttura
        new_h5_path = new_h5_file + ":" + relative_path
        # Aggiorna il percorso nel riferimento
        element.text = new_h5_path

# Sostituisci tutti i riferimenti ai file .h5 con il nuovo file mantenendo la struttura
new_h5_file = "3_Base_PINN_solution.h5"
for elem in root.iter():
    replace_h5_references(elem, new_h5_file)

# Salva il nuovo file XDMF con i riferimenti modificati
new_file_path = "Files/3_Base_PINN_solution.xdmf"
tree.write(new_file_path, encoding="utf-8", xml_declaration=True)